In [13]:
import torch
from typing import Union, List, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from collections import Counter

def build_vocab(iterator, min_freq: int, specials: List[str]):
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)

    idx2word = specials + [
        word for word, cnt in counter.most_common() if cnt >= min_freq
    ]
    word2idx = {word: idx for idx, word in enumerate(idx2word)}
    return word2idx, idx2word

def token_iter(path: str):
    with open(path, encoding='utf-8') as f:
        for line in f:
            yield line.strip().lower().split()


class TextDataset(Dataset):

    special_tokens = ['<pad>', '<unk>', '<sos>', '<eos>']

    pad_id = 0
    unk_id = 1
    sos_id = 2
    eos_id = 3

    def __init__(self, de_path: str, en_path: str, min_freq: int = 3, vocab_de = None, vocab_en = None):
        with open(de_path, encoding='utf-8') as f:
            self.lines_de = f.readlines()
        with open(en_path, encoding='utf-8') as f:
            self.lines_en = f.readlines()

        if vocab_de is None:
            self.word2idx_de, self.idx2word_de = build_vocab(
                token_iter(de_path), min_freq, self.special_tokens
            )
            self.word2idx_en, self.idx2word_en = build_vocab(
                token_iter(en_path), min_freq, self.special_tokens
            )
        else:
            self.word2idx_de, self.idx2word_de = vocab_de
            self.word2idx_en, self.idx2word_en = vocab_en
            
        self.vocab_size_de = len(self.idx2word_de)
        self.vocab_size_en = len(self.idx2word_en)

    def text2ids(self, texts: Union[str, List[str]], lang: str) -> Union[List[int], List[List[int]]]:
        word2idx = self.word2idx_de if lang == 'de' else self.word2idx_en

        is_single = isinstance(texts, str)

        if is_single:
            texts = [texts]
            
        result = [
            [word2idx.get(w, self.unk_id) for w in line.strip().lower().split()]
            for line in texts
        ]
        return result[0] if is_single else result

    def ids2text(self, ids: Union[torch.Tensor, List[int], List[List[int]]], lang: str) -> Union[str, List[str]]:
        
        idx2word = self.idx2word_de if lang == 'de' else self.idx2word_en

        if torch.is_tensor(ids):
            ids = ids.cpu().tolist()

        if not ids:
            return ''

        is_batch = isinstance(ids[0], list)

        def decode(tokens: List[int]) -> str:
            words = []
            for tok in tokens:
                word = idx2word[tok]
                if word == '<eos>':
                    break       
                if word not in ('<sos>', '<pad>'):
                    words.append(word)
            return ' '.join(words)

        if is_batch:
            return [decode(seq) for seq in ids]
        else:
            return decode(ids)

    def __len__(self) -> int:
        return len(self.lines_de)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        de_ids = self.text2ids(self.lines_de[idx], lang='de')
        en_ids = self.text2ids(self.lines_en[idx], lang='en')

        target_ids = [self.sos_id] + en_ids + [self.eos_id]

        return (
            torch.tensor(de_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

def make_batch(batch: List[Tuple[torch.Tensor, torch.Tensor]], pad_id: int = TextDataset.pad_id, additional: int = 5) -> Tuple[torch.Tensor, torch.Tensor]:
    de_ids, en_ids = zip(*batch)

    max_len_de_embed = max(s.size(0) for s in de_ids) + additional
    max_len_en_embed = max(t.size(0) for t in en_ids) + additional

    def pad_sequences(seqs, max_len):
        out = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
        for i, seq in enumerate(seqs):
            length = min(seq.size(0), max_len)
            out[i, :length] = seq[:length]
        return out

    return pad_sequences(de_ids, max_len_de_embed), pad_sequences(en_ids, max_len_en_embed)

def get_dataloader(dataset: TextDataset, batch_size: int, shuffle: bool = True, num_workers: int = 0) -> DataLoader:
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=make_batch,
        pin_memory=True
    )

In [14]:
from torch.utils.data import ConcatDataset

train_ds = TextDataset(
    de_path='/kaggle/input/datasets/shaykoaleksandr/our-data/train.de-en.de',
    en_path='/kaggle/input/datasets/shaykoaleksandr/our-data/train.de-en.en',
    min_freq=3
)
val_ds = TextDataset(
    de_path='/kaggle/input/datasets/shaykoaleksandr/our-data/val.de-en.de',
    en_path='/kaggle/input/datasets/shaykoaleksandr/our-data/val.de-en.en',
    vocab_de=(train_ds.word2idx_de, train_ds.idx2word_de),
    vocab_en=(train_ds.word2idx_en, train_ds.idx2word_en)
)

combined_ds = ConcatDataset([train_ds, val_ds])

combined_dl = DataLoader(combined_ds, batch_size=128, shuffle=True, collate_fn=make_batch)

In [15]:
import math
import torch.nn as nn


class Transformer(nn.Module):

    def __init__(
        self,
        vocab_size_de: int,
        vocab_size_en: int,
        d_model: int = 512,
        n_heads: int = 8,
        n_layers: int = 3,
        d_ff: int = 2048,
        dropout: float = 0.1,
        pad_id: int = 0 
    ):
        super().__init__()

        self.pad_id = pad_id
        self.d_model = d_model

        self.de_embedding = nn.Embedding(vocab_size_de, d_model, padding_idx=pad_id)
        self.en_embedding = nn.Embedding(vocab_size_en, d_model, padding_idx=pad_id)

        pos_embeds = torch.zeros(5000, d_model)
        position = torch.arange(0, 5000).unsqueeze(1).float()
        trig_freq = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pos_embeds[:, 0::2] = torch.sin(position * trig_freq)
        pos_embeds[:, 1::2] = torch.cos(position * trig_freq)
        self.register_buffer('pos_embeds', pos_embeds.unsqueeze(0))

        self.dropout = nn.Dropout(dropout)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        self.output_linear = nn.Linear(d_model, vocab_size_en)

        self.output_linear.weight = self.en_embedding.weight

        self.init_weights()

    def init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def add_positional_encoding(self, x: torch.Tensor) -> torch.Tensor:
        x = x * math.sqrt(self.d_model)
        x = x + self.pos_embeds[:, : x.size(1)]
        return self.dropout(x)

    def make_pad_mask(self, seq: torch.Tensor) -> torch.Tensor:
        return seq == self.pad_id

    def make_decoder_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device), diagonal=1
        ).bool()
        return causal_mask

    def forward(self, de_seq: torch.Tensor, en_seq: torch.Tensor) -> torch.Tensor:
        de_pad_mask   = self.make_pad_mask(de_seq)
        en_pad_mask   = self.make_pad_mask(en_seq)
        en_decoder_mask = self.make_decoder_mask(en_seq.size(1), en_seq.device)

        de_emb = self.add_positional_encoding(self.de_embedding(de_seq))
        en_emb = self.add_positional_encoding(self.en_embedding(en_seq))

        out = self.transformer(
            src=de_emb,
            tgt=en_emb,
            tgt_mask=en_decoder_mask,
            src_key_padding_mask=de_pad_mask,
            tgt_key_padding_mask=en_pad_mask,
            memory_key_padding_mask=de_pad_mask
        )

        return self.output_linear(out)

    @torch.inference_mode()
    def translate(
        self,
        de_seq: torch.Tensor,
        sos_id: int,
        eos_id: int,
        max_len: int = 100
    ) -> list:
    
        self.eval()
    
        de_pad_mask = self.make_pad_mask(de_seq)
        de_emb = self.add_positional_encoding(self.de_embedding(de_seq))
    
        memory = self.transformer.encoder(
            de_emb,
            src_key_padding_mask=de_pad_mask
        )
    
        en_tokens = torch.tensor([[sos_id]], device=de_seq.device) 
        result = []
    
        for _ in range(max_len):
            en_emb = self.add_positional_encoding(self.en_embedding(en_tokens))
            en_decoder_mask = self.make_decoder_mask(en_tokens.size(1), de_seq.device)
    
            out = self.transformer.decoder(
                en_emb,
                memory,
                tgt_mask=en_decoder_mask,
                memory_key_padding_mask=de_pad_mask
            )
    
            next_token = self.output_linear(out[:, -1, :]).argmax(dim=-1).item()
            result.append(next_token)
    
            if next_token == eos_id:
                break
    
            en_tokens = torch.cat([
                en_tokens,
                torch.tensor([[next_token]], device=de_seq.device)
            ], dim=1)
    
        return result

In [16]:
model = Transformer(
    vocab_size_de=train_ds.vocab_size_de,
    vocab_size_en=train_ds.vocab_size_en,
    d_model=512,
    n_heads=8,
    n_layers=1,
    d_ff=2048,
    dropout=0.1,
    pad_id=train_ds.pad_id
)

In [17]:
!pip install sacrebleu

In [18]:
!pip install wandb

In [19]:
import wandb
wandb.login(key='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')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [20]:
from tqdm import tqdm
import sacrebleu
import heapq
import os


def compute_loss(logits, targets, pad_id, smoothing=0.1):
    vocab_size = logits.size(-1)

    logits = logits.reshape(-1, vocab_size)
    targets = targets.reshape(-1)

    loss = nn.functional.cross_entropy(
        logits,
        targets,
        ignore_index=pad_id,
        label_smoothing=smoothing
    )
    return loss

def get_lr(step, d_model, warmup_steps):
    if step == 0:
        step = 1
    return d_model ** (-0.5) * min(step ** (-0.5), step * warmup_steps ** (-1.5))


def postprocess(text):
    words = text.split()
    if not words:
        return text
    
    result = [words[0]]
    for word in words[1:]:
        if word != result[-1]:
            result.append(word)
    return ' '.join(result)


def compute_bleu(model, dataset, device, n_examples=None):
    n = n_examples if n_examples is not None else len(dataset)
    model.eval()

    model_translations = []
    real_translations  = []
    
    print()
    with torch.inference_mode():
        for i in tqdm(range(n), desc='bleu', leave=False):
            de_ids, en_ids = dataset[i]

            de_seq = de_ids.unsqueeze(0).to(device)
            result_ids = model.translate(
                de_seq,
                sos_id=dataset.sos_id,
                eos_id=dataset.eos_id
            )
            if result_ids and result_ids[-1] == dataset.eos_id:
                result_ids = result_ids[:-1]

            en_model_text = dataset.ids2text(result_ids, lang='en')
            en_real_text  = dataset.ids2text(en_ids.tolist(), lang='en')

            model_translations.append(en_model_text)
            real_translations.append(en_real_text)

    bleu = sacrebleu.corpus_bleu(model_translations, [real_translations])
    return bleu.score

def print_examples(model, dataset, device, n_examples=3):
    print()
    with torch.inference_mode():
        for i in range(n_examples):
            de_ids, en_ids = dataset[i]

            de_seq = de_ids.unsqueeze(0).to(device)
            result_ids = model.translate(
                de_seq,
                sos_id=dataset.sos_id,
                eos_id=dataset.eos_id
            )
            if result_ids and result_ids[-1] == dataset.eos_id:
                result_ids = result_ids[:-1]

            de_text = dataset.ids2text(de_ids.tolist(), lang='de')
            en_model_text = dataset.ids2text(result_ids, lang='en')
            en_real_text  = dataset.ids2text(en_ids.tolist(), lang='en')

            print(f'[{i+1}] original_de: {de_text}')
            print(f'model_translation_en: {en_model_text}')
            print(f'original_en: {en_real_text}')
            print()

def train_epoch(model, train_dl, optimizer, scheduler, pad_id, device):
    model.train()
    total_loss   = 0
    total_tokens = 0

    progress = tqdm(train_dl, desc='train', leave=False)

    for de_batch, en_batch in progress:
        de_batch = de_batch.to(device)
        en_batch = en_batch.to(device)

        en_input  = en_batch[:, :-1]
        en_target = en_batch[:, 1:]

        logits = model(de_batch, en_input)
        loss   = compute_loss(logits, en_target, pad_id)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        n_tokens = (en_target != pad_id).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

        progress.set_postfix({
            'loss': f'{loss.item()}',
            'lr':   f'{scheduler.get_last_lr()[0]}'
        })

    return total_loss / total_tokens

@torch.inference_mode()
def val_epoch(model, val_dl, pad_id, device):
    model.eval()
    total_loss = 0
    total_tokens = 0

    progress = tqdm(val_dl, desc='val', leave=False)

    for de_batch, en_batch in progress:
        de_batch = de_batch.to(device)
        en_batch = en_batch.to(device)

        en_input  = en_batch[:, :-1]
        en_target = en_batch[:, 1:]

        logits = model(de_batch, en_input)
        loss = compute_loss(logits, en_target, pad_id)

        n_tokens = (en_target != pad_id).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

        progress.set_postfix({'loss': f'{loss.item()}'})

    return total_loss / total_tokens


def train(model, train_dl, val_dl, train_ds, val_ds,
          n_epochs, pad_id, d_model, warmup_steps, device, patience=5):

    model = model.to(device)

    wandb.init(
        project='de-en-translation',
        config={
            'd_model': d_model,
            'n_heads': 8,
            'n_layers': 1,
            'd_ff': 2048,
            'dropout': 0.1,
            'warmup_steps': warmup_steps,
            'batch_size': train_dl.batch_size,
            'vocab_size_de': train_ds.vocab_size_de,
            'vocab_size_en': train_ds.vocab_size_en,
            'use_validation': val_dl is not None
        })

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1.0,
        betas=(0.9, 0.98),
        eps=1e-9,
        weight_decay=0.01
    )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: get_lr(step, d_model, warmup_steps)
    )

    best_val_loss  = float('inf')
    best_val_bleu  = 0.0
    epochs_without_improvement = 0
    top_checkpoints = []

    for epoch in range(1, n_epochs + 1):

        print(f'Epoch {epoch}/{n_epochs}')
        print()

        train_loss = train_epoch(model, train_dl, optimizer, scheduler, pad_id, device)
        train_ppl  = math.exp(train_loss)

        if val_dl is not None:
            val_loss = val_epoch(model, val_dl, pad_id, device)
            val_ppl  = math.exp(val_loss)

            print_examples(model, val_ds, device)
            val_bleu = compute_bleu(model, val_ds, device, n_examples=300)

            wandb.log({
                'epoch':             epoch,
                'train_loss':        train_loss,
                'train_perplexity':  train_ppl,
                'val_loss':          val_loss,
                'val_perplexity':    val_ppl,
                'val_bleu':          val_bleu,
                'lr':                scheduler.get_last_lr()[0]
            })

            print(
                f'epoch {epoch} | '
                f'train loss {train_loss:.3f} | train ppl {train_ppl:.1f} | '
                f'val loss {val_loss:.3f} | val ppl {val_ppl:.1f} | '
                f'val bleu {val_bleu:.2f} | '
                f'lr {scheduler.get_last_lr()[0]:.6f}'
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), 'best_model.pt')
                print(f'Сохранили лучшую по loss (val loss {val_loss:.3f})')
                wandb.run.summary['best_val_loss'] = val_loss
                wandb.run.summary['best_epoch']    = epoch

            if val_bleu > best_val_bleu:
                best_val_bleu = val_bleu
                epochs_without_improvement = 0
                torch.save(model.state_dict(), 'best_bleu_model.pt')
                print(f'Сохранили лучшую по bleu (val bleu {val_bleu:.2f})')
                wandb.run.summary['best_val_bleu'] = val_bleu
            else:
                epochs_without_improvement += 1
                print(f'нет улучшения {epochs_without_improvement}/{patience} эпох')

        else:
            wandb.log({
                'epoch':            epoch,
                'train_loss':       train_loss,
                'train_perplexity': train_ppl,
                'lr':               scheduler.get_last_lr()[0]
            })

            print(
                f'epoch {epoch} | '
                f'train loss {train_loss:.3f} | train ppl {train_ppl:.1f} | '
                f'lr {scheduler.get_last_lr()[0]:.6f}'
            )

        checkpoint_path = f'checkpoint_epoch_{epoch}.pt'
        torch.save(model.state_dict(), checkpoint_path)
        heapq.heappush(top_checkpoints, (val_bleu if val_dl is not None else train_loss * -1, checkpoint_path))

        if len(top_checkpoints) > 5:
            worst, check_path = heapq.heappop(top_checkpoints)
            os.remove(check_path)
            print(f'Удалили чекпоинт {check_path}')

        if val_dl is not None and epochs_without_improvement >= patience:
            print(f'early stopping на эпохе {epoch}')
            break

    print(f'\nУсредняем топ-{len(top_checkpoints)} чекпоинтов...')

    _, first_path = top_checkpoints[0]
    avg_params = torch.load(first_path, map_location=device)

    for _, path in top_checkpoints[1:]:
        state = torch.load(path, map_location=device)
        for key in avg_params:
            avg_params[key] += state[key]

    n = len(top_checkpoints)
    for key in avg_params:
        avg_params[key] = avg_params[key] / n

    model.load_state_dict(avg_params)
    torch.save(avg_params, 'averaged_model.pt')
    print('Сохранили усреднённую модель')

    if val_ds is not None:
        val_loss_avg = val_epoch(model, val_dl, pad_id, device) if val_dl is not None else None
        print('\nПримеры переводов усреднённой модели')
        val_bleu_avg = compute_bleu(model, val_ds, device, n_examples=300)

        if val_loss_avg is not None:
            wandb.run.summary['averaged_val_loss'] = val_loss_avg
        wandb.run.summary['averaged_val_bleu'] = val_bleu_avg

        print(f'Усреднённая модель: val bleu {val_bleu_avg:.2f}')

    wandb.finish()

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'устройство: {device}')

train(
    model=model,
    train_dl=combined_dl,
    val_dl=None,
    train_ds=train_ds,
    val_ds=val_ds,
    n_epochs=15,
    pad_id=train_ds.pad_id,
    d_model=512,
    warmup_steps=8000,
    device=device,
    patience=16
)

устройство: cuda


Epoch 1/15



epoch 1 | train loss 6.853 | train ppl 946.9 | lr 0.000095
Epoch 2/15



epoch 2 | train loss 4.876 | train ppl 131.1 | lr 0.000190
Epoch 3/15



epoch 3 | train loss 4.145 | train ppl 63.1 | lr 0.000285
Epoch 4/15



...

epoch 14 | train loss 2.920 | train ppl 18.5 | lr 0.000301
Удалили чекпоинт checkpoint_epoch_9.pt
Epoch 15/15



epoch 15 | train loss 2.885 | train ppl 17.9 | lr 0.000291
Удалили чекпоинт checkpoint_epoch_10.pt

Усредняем топ-5 чекпоинтов...
Сохранили усреднённую модель

Примеры переводов усреднённой модели



That's 100 lines that end in a tokenized period ('.')  
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Усреднённая модель: val bleu 28.43


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▃▄▆██▇▇▆▆▆▅▅▅▅
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
train_perplexity,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁
averaged_val_bleu,28.42608
epoch,15
lr,0.00029
train_loss,2.88473
train_perplexity,17.89879


In [ ]:
model.load_state_dict(torch.load('/kaggle/working/averaged_model.pt', map_location=device))
model.eval()
test_path = '/kaggle/input/datasets/shaykoaleksandr/our-data/test1.de-en.de'

with open(test_path, encoding='utf-8') as f:
    test_lines = f.readlines()

translations = []

for i, line in enumerate(tqdm(test_lines, desc='перевод')):
    de_ids = train_ds.text2ids(line.strip(), lang='de')
    de_seq = torch.tensor([de_ids], dtype=torch.long).to(device)

    result_ids = model.translate(
        de_seq,
        sos_id=train_ds.sos_id,
        eos_id=train_ds.eos_id
    )

    if result_ids and result_ids[-1] == train_ds.eos_id:
        result_ids = result_ids[:-1]

    translation = train_ds.ids2text(result_ids, lang='en') 
    translations.append(translation)

    if i < 5:
        print(f'de_text: {line.strip()}')
        print(f'translation: {translation}')

with open('answer.txt', 'w', encoding='utf-8') as f:
    for translation in translations:
        f.write(translation + '\n')

print(f'\nсохранили {len(translations)}')